# Beginner Machine Learning Pipelines with Scikit-learn

This lab shows the full beginner machine-learning workflow: load data, clean data, explore data, choose features, split train/test data, train a model, evaluate it, export it, and test it on one new data point.

## Important: Use a Python Notebook Runtime

Run this notebook with a **Python 3** kernel in Jupyter, VS Code, JupyterLab, or Google Colab. Do not run these cells in SQL Server, DBeaver, Azure Data Studio SQL query mode, or `sqlcmd`.

If you see SQL Server messages such as `Incorrect syntax near '%'`, the notebook code is being executed by a SQL engine instead of Python.

These beginner notebooks (`05`, `06`, and `07`) do **not** require a database connection. They load `data/beginner_financial_indicators.csv` when the file is present, and automatically create a small fallback dataset when the file is not present, which makes them work in Google Colab.


## 0. Setup

Use this setup cell only inside a Python notebook. `%pip` is valid in Jupyter and Google Colab, but it is not SQL syntax.


In [6]:
%pip install pandas numpy scipy matplotlib seaborn scikit-learn joblib -q

import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

DATA_PATH = Path("data/beginner_financial_indicators.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def make_fallback_dataset():
    """Create a small dataset only when the CSV file is not available, such as in Colab."""
    dates = pd.date_range("2026-01-01", periods=10, freq="MS")
    templates = [
        ("Bank A", "Maseru", "Commercial Bank", 36.0, 19.5, 3.1, 5.0, 980, 610, 3100),
        ("Bank B", "Leribe", "Commercial Bank", 31.5, 17.4, 4.8, 4.2, 860, 590, 2500),
        ("Bank C", "Mafeteng", "Microfinance", 27.0, 14.8, 6.5, 3.1, 520, 380, 1700),
        ("Bank D", "Quthing", "Microfinance", 23.0, 12.4, 8.2, 2.2, 430, 350, 1350),
    ]
    rows = []
    for month_index, date in enumerate(dates):
        for inst_index, item in enumerate(templates):
            name, region, inst_type, liq, cap, npl, profit, dep, loans, tx = item
            liquidity = liq - month_index * (0.35 + inst_index * 0.08)
            capital = cap - month_index * (0.10 + inst_index * 0.02)
            npl_ratio = npl + month_index * (0.18 + inst_index * 0.05)
            stress = int((liquidity < 24) or (capital < 12) or (npl_ratio > 8.8))
            rows.append({
                "date": date,
                "institution": name,
                "region": region,
                "institution_type": inst_type,
                "liquidity_ratio": round(liquidity, 2),
                "capital_adequacy_ratio": round(capital, 2),
                "npl_ratio": round(npl_ratio, 2),
                "profit_margin": round(profit - npl_ratio * 0.12, 2),
                "total_deposits_m": dep + month_index * (8 - inst_index * 2),
                "total_loans_m": loans + month_index * (6 + inst_index * 2),
                "transaction_count": tx + month_index * (55 - inst_index * 5),
                "stress_flag": stress,
            })
    return pd.DataFrame(rows)


if DATA_PATH.exists():
    # parse_dates tells pandas that the date column should behave like a date, not plain text.
    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
else:
    df = make_fallback_dataset()

print("Rows loaded:", len(df))
display(df.head())

Msg 102, Level 15, State 1, Line 1
Incorrect syntax near '%'.
Msg 103, Level 15, State 4, Line 30
The identifier that starts with '
        ("Bank A", "Maseru", "Commercial Bank", 36.0, 19.5, 3.1, 5.0, 980, 610, 3100),
        ("Bank B", "Leribe", "Commercial' is too long. Maximum length is 128.
Msg 1038, Level 15, State 4, Line 36
An object or column name is missing or empty. For SELECT INTO statements, verify each column has a name. For other statements, look for empty alias names. Aliases defined as "" or [] are not allowed. Change the alias to a valid name.
Msg 178, Level 15, State 1, Line 58
A RETURN statement with a return value cannot be used in this context.
Msg 156, Level 15, State 1, Line 61
Incorrect syntax near the keyword 'exists'.
Msg 128, Level 15, State 1, Line 67
The name "Rows loaded:" is not permitted in this context. Valid expressions are constants, constant expressions, and (in some contexts) variables. Column names are not permitted.

Total execution time: 00:00:00.004

## 1. Load and Clean the Dataset

A model learns patterns from rows and columns. The target is the value we want to predict. The features are the input columns used to make the prediction.

In [ ]:
print("Rows before cleaning:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print("Missing values before cleaning:")
display(df.isna().sum())

# Work on a copy so the raw data stays unchanged.
clean_df = df.drop_duplicates().copy()

numeric_cols = [
    "liquidity_ratio",
    "capital_adequacy_ratio",
    "npl_ratio",
    "profit_margin",
    "total_deposits_m",
    "total_loans_m",
    "transaction_count",
]

# Convert numeric columns safely. Bad values become NaN so they can be filled.
for column in numeric_cols:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

# Fill missing numeric values with the median. The median is simple and less affected by outliers.
clean_df[numeric_cols] = clean_df[numeric_cols].fillna(clean_df[numeric_cols].median())

print("Rows after cleaning:", len(clean_df))
print("Missing values after cleaning:")
display(clean_df.isna().sum())
display(clean_df.head())

## 2. Simple EDA Before Modelling

Before training a model, inspect the data. This helps you choose useful features and avoid obvious data problems.

In [ ]:
display(clean_df[numeric_cols + ["stress_flag"]].describe().T.round(2))

# Compare averages for stressed and non-stressed rows.
stress_summary = clean_df.groupby("stress_flag")[numeric_cols].mean().round(2)
display(stress_summary)

## 3. Classification Pipeline: Predict Stress Flag

Classification predicts a category. Here the model predicts whether a row is stressed (`1`) or not stressed (`0`).

In [ ]:
# Step 1: choose feature columns and target column.
classification_features = [
    "liquidity_ratio",
    "capital_adequacy_ratio",
    "npl_ratio",
    "profit_margin",
    "total_loans_m",
]
X = clean_df[classification_features]
y = clean_df["stress_flag"]

# Step 2: split rows into training data and testing data.
# stratify=y keeps both stress classes represented in train and test data.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

# Step 3: build the pipeline.
# StandardScaler puts numeric columns on a similar scale.
# LogisticRegression is a simple classification model.
classification_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

# Step 4: train the model on the training rows.
classification_model.fit(X_train, y_train)

# Step 5: test the model on rows it did not train on.
classification_predictions = classification_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, classification_predictions), 3))
print(classification_report(y_test, classification_predictions))

## 4. Classification Evaluation: Confusion Matrix

The confusion matrix shows correct and incorrect predictions. Rows are actual values. Columns are predicted values.

In [ ]:
cm = confusion_matrix(y_test, classification_predictions)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"],
)
plt.title("Stress Classifier Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "beginner_ml_confusion_matrix.png", dpi=150)
plt.show()

## 5. Export and Test the Classification Model

Exporting the model lets another script or notebook load it later and use it on new data.

In [ ]:
classification_model_path = OUTPUT_DIR / "beginner_stress_classifier.joblib"
dump(classification_model, classification_model_path)

loaded_classifier = load(classification_model_path)

# This is one new row with the same feature columns used during training.
new_observation = pd.DataFrame([{
    "liquidity_ratio": 22.5,
    "capital_adequacy_ratio": 11.8,
    "npl_ratio": 9.2,
    "profit_margin": 1.4,
    "total_loans_m": 410.0,
}])

new_prediction = loaded_classifier.predict(new_observation)[0]
stress_probability = loaded_classifier.predict_proba(new_observation)[0][1]

print("Predicted stress flag:", new_prediction)
print("Stress probability:", round(stress_probability, 3))
print("Saved model:", classification_model_path)

## 6. Regression Pipeline: Predict Profit Margin

Regression predicts a number. Here the model predicts profit margin from liquidity, capital, NPL ratio, deposits, loans, and transactions.

In [ ]:
# Step 1: choose features and target.
regression_features = [
    "liquidity_ratio",
    "capital_adequacy_ratio",
    "npl_ratio",
    "total_deposits_m",
    "total_loans_m",
    "transaction_count",
]
X = clean_df[regression_features]
y = clean_df["profit_margin"]

# Step 2: split the data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Step 3: build and train the model.
regression_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
regression_model.fit(X_train, y_train)

# Step 4: evaluate predictions.
regression_predictions = regression_model.predict(X_test)
mae = mean_absolute_error(y_test, regression_predictions)
rmse = np.sqrt(mean_squared_error(y_test, regression_predictions))
r2 = r2_score(y_test, regression_predictions)

print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R2:", round(r2, 3))

## 7. Export and Test the Regression Model

In [ ]:
regression_model_path = OUTPUT_DIR / "beginner_profit_regression.joblib"
dump(regression_model, regression_model_path)

loaded_regression = load(regression_model_path)

new_regression_point = pd.DataFrame([{
    "liquidity_ratio": 30.0,
    "capital_adequacy_ratio": 16.5,
    "npl_ratio": 5.2,
    "total_deposits_m": 900.0,
    "total_loans_m": 650.0,
    "transaction_count": 2600,
}])

predicted_profit = loaded_regression.predict(new_regression_point)[0]
print("Predicted profit margin:", round(predicted_profit, 2))
print("Saved model:", regression_model_path)

## 8. Simple Time-Series Style Prediction

A beginner time-series model can use time order as a feature. This is not advanced forecasting, but it shows the basic idea: train on earlier dates and test on later dates.

In [ ]:
bank_a = clean_df[clean_df["institution"] == "Bank A"].sort_values("date").copy()

# Convert dates into a simple number: 0, 1, 2, 3, and so on.
bank_a["month_number"] = np.arange(len(bank_a))

X = bank_a[["month_number"]]
y = bank_a["liquidity_ratio"]

# For time-ordered data, do not shuffle. Train on the past and test on the future.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, shuffle=False)

time_model = LinearRegression()
time_model.fit(X_train, y_train)
time_predictions = time_model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, time_predictions), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, time_predictions)), 3))

future_months = pd.DataFrame({"month_number": [len(bank_a), len(bank_a) + 1, len(bank_a) + 2]})
future_predictions = time_model.predict(future_months)

future_result = pd.DataFrame({
    "future_month_number": future_months["month_number"],
    "predicted_liquidity_ratio": np.round(future_predictions, 2),
})
display(future_result)

## 9. Export Time Model and Metrics

In [ ]:
time_model_path = OUTPUT_DIR / "beginner_liquidity_time_model.joblib"
dump(time_model, time_model_path)

metrics = pd.DataFrame([
    {"model": "classification_logistic_regression", "metric": "accuracy", "value": round(accuracy_score(clean_df["stress_flag"], classification_model.predict(clean_df[classification_features])), 3)},
    {"model": "regression_linear_regression", "metric": "mae", "value": round(mae, 3)},
    {"model": "regression_linear_regression", "metric": "rmse", "value": round(rmse, 3)},
    {"model": "time_linear_regression", "metric": "test_mae", "value": round(mean_absolute_error(y_test, time_predictions), 3)},
])

metrics_path = OUTPUT_DIR / "beginner_ml_model_metrics.csv"
metrics.to_csv(metrics_path, index=False)

print("Saved metrics:", metrics_path)
print("Saved time model:", time_model_path)
display(metrics)

## Key Pipeline Pattern

Use this same simple pattern for most beginner models:

1. Load the dataset.
2. Clean missing values and duplicates.
3. Explore the data.
4. Choose features and target.
5. Split into train and test sets.
6. Train the model.
7. Evaluate on the test set.
8. Export the model.
9. Load the model and test it on one new data point.